### Resolucion Laboratorio 1

In [26]:
import pandas as pd 

In [27]:
# leer dataset
df = pd.read_csv("https://raw.githubusercontent.com/fvillena/biocompu/2024/data/drug.csv")


In [28]:
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,DrugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,DrugY


In [29]:
#vemos la cantidad de atributos que tiene el conjunto de datos
df.shape 

(200, 6)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          200 non-null    int64  
 1   Sex          200 non-null    object 
 2   BP           200 non-null    object 
 3   Cholesterol  200 non-null    object 
 4   Na_to_K      200 non-null    float64
 5   Drug         200 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 9.5+ KB


In [31]:
df.describe()

,Age,Na_to_K
count,200.000000,200.000000
mean,44.315000,16.084485
std,16.544315,7.223956
min,15.000000,6.269000
25%,31.000000,10.445500
50%,45.000000,13.936500
75%,58.000000,19.380000
max,74.000000,38.247000


In [32]:
#vemos los datos nulos
df.isnull().sum()

Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
Drug           0
dtype: int64

In [33]:
#vemos la cantidad valores por cada valor de la variable objetivo
df["Drug"].value_counts()

Drug
DrugY    91
drugX    54
drugA    23
drugC    16
drugB    16
Name: count, dtype: int64

In [34]:
#aplicamos la tecnica de one-hot encoding
cols_categoricas = ['Sex', 'BP', 'Cholesterol']

df_encoded = pd.get_dummies(df, columns=cols_categoricas)
df_encoded.head()

,Age,Na_to_K,Drug,Sex_F,Sex_M,BP_HIGH,BP_LOW,BP_NORMAL,Cholesterol_HIGH,Cholesterol_NORMAL
0,23,25.355,DrugY,True,False,True,False,False,True,False
1,47,13.093,drugC,False,True,False,True,False,True,False
2,47,10.114,drugC,False,True,False,True,False,True,False
3,28,7.798,drugX,True,False,False,False,True,True,False
4,61,18.043,DrugY,True,False,False,True,False,True,False


In [35]:
#separamos los conjuntos de entrenamiento y test
from sklearn.model_selection import train_test_split

X = df_encoded.drop("Drug", axis=1) 
y = df_encoded['Drug']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)


In [36]:
#entrenamos los 3 modelos
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Modelo 1
model1 = SVC(C=1, gamma=0.1, kernel='rbf')
model1.fit(X_train, y_train)


# Modelo 2
model2 = SVC(C=10, gamma=0.01, kernel='rbf')
model2.fit(X_train, y_train)


# Modelo 3
model3 = SVC(C=0.1, gamma=1, kernel='linear')
model3.fit(X_train, y_train)



SVC(C=0.1, gamma=1, kernel='linear')

In [37]:
#prediccion sobre conjunto test
# Modelo 1
y_pred1 = model1.predict(X_test)

# Modelo 2
y_pred2 = model2.predict(X_test)

# Modelo 3
y_pred3 = model3.predict(X_test)


In [38]:
#calcular al menos dos metricas

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def metricas(y_test, y_pred, modelo):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Modelo {modelo}:")
    print(f"Exactitud: {accuracy:.2f}")
    print(f"Precisión: {precision:.2f}")
    print(f"Sensibilidad (Recall): {recall:.2f}")
    print(f"F1-score: {f1:.2f}\n")

    return accuracy,precision,recall,f1
    

In [39]:
def metricas_por_clase(y_test, y_pred, modelo):
    precision = precision_score(y_test, y_pred, average=None, labels=y_test.unique())
    recall = recall_score(y_test, y_pred, average=None, labels=y_test.unique())
    f1 = f1_score(y_test, y_pred, average=None, labels=y_test.unique())
    
    print(f"Modelo {modelo}:")
    for i, label in enumerate(y_test.unique()):
        print(f"Clase {label}:")
        print(f"  Precisión: {precision[i]:.2f}")
        print(f"  Sensibilidad (Recall): {recall[i]:.2f}")
        print(f"  F1-score: {f1[i]:.2f}\n")

In [40]:
accuracy1,precision1, recall1, f1_1 =metricas(y_test, y_pred1, 1)

Modelo 1:
Exactitud: 0.70
Precisión: 0.64
Sensibilidad (Recall): 0.70
F1-score: 0.66



/home/vscode/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
metricas_por_clase(y_test, y_pred1, 1)

Modelo 1:
Clase DrugY:
  Precisión: 0.94
  Sensibilidad (Recall): 1.00
  F1-score: 0.97

Clase drugX:
  Precisión: 0.38
  Sensibilidad (Recall): 0.67
  F1-score: 0.48

Clase drugC:
  Precisión: 0.50
  Sensibilidad (Recall): 0.25
  F1-score: 0.33

Clase drugB:
  Precisión: 0.00
  Sensibilidad (Recall): 0.00
  F1-score: 0.00

Clase drugA:
  Precisión: 0.33
  Sensibilidad (Recall): 0.17
  F1-score: 0.22



/home/vscode/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [42]:
#modelo 2
accuracy2,precision2, recall2, f1_2=metricas(y_test, y_pred2, 2)

Modelo 2:
Exactitud: 0.83
Precisión: 0.84
Sensibilidad (Recall): 0.83
F1-score: 0.83



In [43]:
metricas_por_clase(y_test, y_pred2, 2)

Modelo 2:
Clase DrugY:
  Precisión: 0.97
  Sensibilidad (Recall): 1.00
  F1-score: 0.98

Clase drugX:
  Precisión: 0.59
  Sensibilidad (Recall): 0.83
  F1-score: 0.69

Clase drugC:
  Precisión: 0.50
  Sensibilidad (Recall): 0.25
  F1-score: 0.33

Clase drugB:
  Precisión: 1.00
  Sensibilidad (Recall): 0.67
  F1-score: 0.80

Clase drugA:
  Precisión: 0.75
  Sensibilidad (Recall): 0.50
  F1-score: 0.60



In [44]:
#modelo 3
accuracy3,precision3, recall3, f1_3=metricas(y_test, y_pred3, 3)

Modelo 3:
Exactitud: 1.00
Precisión: 1.00
Sensibilidad (Recall): 1.00
F1-score: 1.00



In [45]:
metricas_por_clase(y_test, y_pred3, 3)

Modelo 3:
Clase DrugY:
  Precisión: 1.00
  Sensibilidad (Recall): 1.00
  F1-score: 1.00

Clase drugX:
  Precisión: 1.00
  Sensibilidad (Recall): 1.00
  F1-score: 1.00

Clase drugC:
  Precisión: 1.00
  Sensibilidad (Recall): 1.00
  F1-score: 1.00

Clase drugB:
  Precisión: 1.00
  Sensibilidad (Recall): 1.00
  F1-score: 1.00

Clase drugA:
  Precisión: 1.00
  Sensibilidad (Recall): 1.00
  F1-score: 1.00



In [46]:
# Comparar los modelos y seleccionar el mejor


metrics = {
    "Modelo 1": {"precision": precision1, "recall": recall1, "f1": f1_1},
    "Modelo 2": {"precision": precision2, "recall": recall2, "f1": f1_2},
    "Modelo 3": {"precision": precision3, "recall": recall3, "f1": f1_3}
}

best_model = max(metrics, key=lambda x: metrics[x]['f1'])

print("Métricas de rendimiento ponderadas para cada modelo:")
for model, scores in metrics.items():
    print(f"{model}:")
    print(f"  Precisión: {scores['precision']:.2f}")
    print(f"  Recall: {scores['recall']:.2f}")
    print(f"  F1-score: {scores['f1']:.2f}\n")

print(f"El mejor modelo es {best_model} basado en la F1-score.")

Métricas de rendimiento ponderadas para cada modelo:
Modelo 1:
  Precisión: 0.64
  Recall: 0.70
  F1-score: 0.66

Modelo 2:
  Precisión: 0.84
  Recall: 0.83
  F1-score: 0.83

Modelo 3:
  Precisión: 1.00
  Recall: 1.00
  F1-score: 1.00

El mejor modelo es Modelo 3 basado en la F1-score.


## Una manera mas resumida

aprovechemos las funciones que nos da el paquete de sklearn para probar varios modelos

In [47]:
from sklearn.model_selection import GridSearchCV

# Definir el modelo SVM
svc = SVC()

# Definir la cuadrícula de hiperparámetros
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': [1, 0.1, 0.01],
    'kernel': ['rbf', 'linear']
}

# Crear el objeto GridSearchCV
grid = GridSearchCV(svc, param_grid, refit=True, verbose=2, cv=5)

# Entrenar el modelo
grid.fit(X_train, y_train)

# Imprimir los mejores parámetros encontrados por GridSearchCV
print("Mejores hiperparámetros encontrados:")
print(grid.best_params_)

# Obtener las predicciones de todos los modelos ajustados
results = pd.DataFrame(grid.cv_results_)
for i, params in enumerate(results['params']):
    print(f"\nModelo {i+1} con parámetros: {params}")
    model = SVC(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"Reporte de clasificación para el modelo {i+1}:")
    print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   0.0s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   0.0s
[CV] END .......................C=0.1, gamma=0.1

/home/vscode/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/vscode/.local/li